In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
%load_ext autoreload
%autoreload 2

import json
import numpy as np
from sources.MKP import MultipleKnapsackProblem
from sources.MKPgrover import MKPGrover
from matplotlib import pyplot as plt

class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super().default(obj)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:

## LOAD DATA ##

# Load instances
with open('data/small_instances.json', 'r') as f:
    instances = json.load(f)

## LOAD larger instances ##

with open('data/larger_instances.json', 'r') as f:
  data = json.load(f)

scenarios = []
maxima = []
no_variables = []
for i in range(len(data)):
  values = data[i]["article_reward"]
  weights = data[i]["article_weight"]
  capacities = data[i]["knapsack_capacity"]
  maximum = data[i]["optimal_solution"]
  maxima.append(maximum)
  no_variables.append(len(weights)*len(capacities))
  # set 75% optimal as the intial threshold
  initial_threshold = int(np.ceil(0.75*maximum))
  scenarios.append({"instance": MultipleKnapsackProblem(values, weights, capacities), "threshold": initial_threshold})

# lbitstrings
with open('data/bitstrings_75.json') as f:
    mkp_bitstrings = json.load(f)



In [ ]:
# Grover experiments for different branching criteria

L = 5/4 # parameter for growth of grover iterations
runs = 20 # number of runs per configuration
nbh_weights = [i/10 for i in range(11)]
branching_configs = [[1,0,0],[0,1,0],[0,0,1]]
results={}
for nbh_weight in nbh_weights:
    for obj_weight in [i/10 for i in range(int(11-nbh_weight*10))]:
        tb_weight = 1-(nbh_weight + obj_weight)
        branching_weights = [nbh_weight, obj_weight, tb_weight]
        print("weights: ", branching_weights)
        results[str(branching_weights)] = {}
        for i,s in enumerate(scenarios):
            mkp = s["instance"]
            thresh = s["threshold"]
            aa_solver = MKPGrover(mkp)
            results[str(branching_weights)][i] = {}
            for r in range(runs):
                results[str(branching_weights)][i][r] = {}
                greedy = aa_solver.greedy_sol
                aa_solver.set_branching_probs(branching_weights=branching_weights, current_sol=greedy)
                sol, val, total_iterations, threshold_updates = aa_solver.iterative_solver_gas(initial_threshold=thresh, bitstrings=mkp_bitstrings[i], L=L, branching_weights=branching_weights)
                results[str(branching_weights)][i][r]["solution"] = sol
                results[str(branching_weights)][i][r]["approx"] = val/maxima[i]
                print("approx: ", val/maxima[i])
                results[str(branching_weights)][i][r]["iterations"] = total_iterations
                results[str(branching_weights)][i][r]["updates"] = threshold_updates
with open(f"results/Grover/mixed_branching_linear_termination.json", "w") as outfile:
    json.dump(results, outfile, indent=4, cls=NumpyEncoder)


weights:  [0.0, 0.0, 1.0]
no improvement found
improvement found
improvement found
improvement found
no improvement found
no improvement found
no improvement found
no improvement found
no improvement found
no improvement found
no improvement found
no improvement found
no improvement found
improvement found
no improvement found
no improvement found
no improvement found
no improvement found
no improvement found
no improvement found
no improvement found
no improvement found
no improvement found
no improvement found
no improvement found
approx:  0.9625
no improvement found
improvement found
improvement found
no improvement found
no improvement found
no improvement found
improvement found
improvement found
no improvement found
no improvement found
no improvement found
no improvement found
improvement found
no improvement found
no improvement found
no improvement found
no improvement found
no improvement found
no improvement found
no improvement found
no improvement found
approx:  0.9625
imp

Traceback (most recent call last):
  File "C:\Users\hessmaximili\AppData\Roaming\Python\Python310\site-packages\IPython\core\interactiveshell.py", line 3378, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "C:\Users\hessmaximili\AppData\Local\Temp\ipykernel_32304\3709276823.py", line 21, in <module>
    sol, val, total_iterations, threshold_updates = aa_solver.iterative_solver_gas(initial_threshold=thresh, bitstrings=mkp_bitstrings[i], L=L, branching_weights=branching_weights)
  File "c:\Users\hessmaximili\Thesis_code\MKP\sources\MKPgrover.py", line 59, in iterative_solver_gas
    pre_grover_probs = [self.bitstring_prob(bitstring) for bitstring in sols]
  File "c:\Users\hessmaximili\Thesis_code\MKP\sources\MKPgrover.py", line 59, in <listcomp>
    pre_grover_probs = [self.bitstring_prob(bitstring) for bitstring in sols]
  File "c:\Users\hessmaximili\Thesis_code\MKP\sources\MKPgrover.py", line -1, in bitstring_prob
KeyboardInterrupt

During handling of the above